### author by yangshichen
### 注意：脚本仅供参考，使用前请仔细阅读

In [1]:
import os
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import seaborn as sb
import seaborn as sns
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt
from scipy.io import mmread
from scipy.sparse import csr_matrix
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
from scipy.sparse import issparse
import decoupler as dc
from sklearn.preprocessing import QuantileTransformer
from multiprocessing import Pool, cpu_count
from functools import partial
from sklearn.preprocessing import QuantileTransformer, StandardScaler
quantile_transformer = QuantileTransformer(output_distribution='normal', random_state=0)
scaler = StandardScaler()

import warnings
warnings.filterwarnings("ignore")

In [2]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, frameon=False)
sc._settings.ScanpyConfig.n_jobs=70

scanpy==1.9.8 anndata==0.9.2 umap==0.5.7 numpy==1.24.4 scipy==1.10.1 pandas==2.0.3 scikit-learn==1.3.2 statsmodels==0.14.1 pynndescent==0.5.13


In [3]:
import os
os.environ["R_HOME"] = "/home/yangshichen//mambaforge/envs/QTL/lib/R"
os.environ["R_LIBS_USER"] = "/home/yangshichen/mambaforge/envs/QTL/lib/R/library"
import pandas as pd
import torch
import tensorqtl
from tensorqtl import  cis
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch: {torch.__version__} (CUDA {torch.version.cuda}), device: {device}")
print(f"pandas: {pd.__version__}")

torch: 2.4.1+cu121 (CUDA 12.1), device: cpu
pandas: 2.0.3


### Use tensorqtl-eQTL

#### Gene expression.csv转bed

In [4]:
# 输入目录、输出目录
input_dir = "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/scRNA-seq/02.normal_dis/"
output_dir1 = "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/scRNA-seq/06.gene_expression_bed/"
output_dir2 = "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/scRNA-seq/07.gene_expression_csv/"

# 基因注释表（必须包含 gene_id, chrom, start, end）
annotation_df = pd.read_csv("/media/AnalysisDisk2/Yangshichen/0_HIV_RNA/QTL/Data/Genotype/gene_annotation.txt", sep='\t')
annotation_df = annotation_df.rename(columns={
    "gene_id": "gene_id",
    "chr": "chr",
    "left": "start",
    "right": "end"})
annotation_df

,gene_id,chr,start,end
0,DDX11L17,chr1,182696,184174
1,PRDM16,chr1,3069168,3438621
2,PEX10,chr1,2403964,2413797
3,RPL21P21,chr1,10054445,10054781
4,LINC01345,chr1,3944547,3949024
...,...,...,...,...
41008,ZNF736P9Y,chrY,8068896,8070723
41009,FAM197Y8,chrY,9354846,9356814
41010,PRKY,chrY,7274313,7371868
41011,SEPTIN14P23,chrY,25378300,25394719


In [5]:
import pandas as pd
import re
def chrom_sort_key(chrom):
    # 提取数字部分（chr1 → 1, chrX → 23, chrY → 24）
    match = re.match(r'chr(\d+)$', chrom)
    if match:
        return int(match.group(1))
    elif chrom == 'chrX':
        return 23
    elif chrom == 'chrY':
        return 24
    elif chrom == 'chrM':
        return 25
    else:
        return 100  # 把奇怪的染色体排到后面

In [6]:
#遍历所有csv文件
for fname in os.listdir(input_dir):
    if fname.endswith(".csv"):
        print("Processing:", fname)
        expr_df = pd.read_csv(os.path.join(input_dir, fname), index_col=0).T
        expr_df.index.name = 'gene_id'
        expr_df.reset_index(inplace=True)

        #合并注释
        merged_df = annotation_df.merge(expr_df, on='gene_id')
        merged_df = merged_df[['chr', 'start', 'end', 'gene_id'] + list(expr_df.columns[1:])]

        #按照染色体顺序排序（否则会报错）
        autosomes = [f"chr{i}" for i in range(1, 23)]
        merged_df = merged_df[merged_df["chr"].isin(autosomes)]
        merged_df['chr_sort_key'] = merged_df['chr'].apply(chrom_sort_key)
        merged_df = merged_df.sort_values(by=['chr_sort_key', 'start', 'end']).drop(columns='chr_sort_key').reset_index(drop=True)

        #保存为bed和csv格式
        bed_fname = fname.replace(".csv", ".bed")
        merged_df.to_csv(os.path.join(output_dir1, bed_fname), sep='\t', index=False)
        merged_df.to_csv(os.path.join(output_dir2, fname), index=False)
        print("Processing:", fname, "Done!")

Processing: Adaptive NK cells.csv
Processing: Adaptive NK cells.csv Done!
Processing: ALPL- MARCKS- NDNs.csv
Processing: ALPL- MARCKS- NDNs.csv Done!
Processing: ASDC.csv
Processing: ASDC.csv Done!
Processing: Atypical naïve B cells.csv
Processing: Atypical naïve B cells.csv Done!
Processing: Basophils.csv
Processing: Basophils.csv Done!
Processing: CCR4+ CD8+ Tcm.csv
Processing: CCR4+ CD8+ Tcm.csv Done!
Processing: CCR4- CD8+ Tcm.csv
Processing: CCR4- CD8+ Tcm.csv Done!
Processing: CD14+ cDC2.csv
Processing: CD14+ cDC2.csv Done!
Processing: CD177int iLDNs.csv
Processing: CD177int iLDNs.csv Done!
Processing: CD1C+ cDC2.csv
Processing: CD1C+ cDC2.csv Done!
Processing: CD27+ IgD+ atypical memory B cells.csv
Processing: CD27+ IgD+ atypical memory B cells.csv Done!
Processing: CD27+ IgD- atypical memory B cells.csv
Processing: CD27+ IgD- atypical memory B cells.csv Done!
Processing: CD27+ MAIT.csv
Processing: CD27+ MAIT.csv Done!
Processing: CD27+ Th1.csv
Processing: CD27+ Th1.csv Done!
Pr

In [7]:
merged_df

,chr,start,end,gene_id,HP-253,HP-778,HP-263,HP-431,HP-869,HP-111,...,HP-89,HP-384,HP-639,HP-697,HP-283,HP-237,HP-282,HP-404,HP-262,HP-825
0,chr1,825138,859446,LINC01128,0.699345,0.436828,-1.887016,1.105567,0.728943,1.280193,...,-1.887016,1.345523,-1.887016,1.024452,0.195584,0.880230,0.146236,0.601588,-1.887016,0.455934
1,chr1,944203,959309,NOC2L,0.514083,0.106399,0.498417,0.851569,-0.241444,-3.299698,...,-3.299698,0.945597,1.214452,-0.450254,0.232667,0.124648,0.220339,0.212140,0.398398,0.120593
2,chr1,960584,965719,KLHL17,0.172558,-1.761082,-1.761082,-1.761082,0.117145,-1.761082,...,-1.761082,0.360170,0.906860,0.172558,0.888675,0.495955,0.621252,1.126121,-1.761082,0.615082
3,chr1,1001138,1014540,ISG15,-0.078250,-0.474597,0.503714,1.185523,-0.662264,0.503714,...,1.504617,-0.051069,-0.351765,-0.563323,-0.566723,-0.029380,-0.296077,-0.951954,-0.048355,-0.418243
4,chr1,1081818,1116361,C1orf159,1.136246,0.202413,-1.926320,0.473467,0.489175,-1.926320,...,-1.926320,-1.926320,0.473467,0.121069,0.255582,-0.034023,0.289105,0.206556,1.247201,0.362476
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8508,chr22,50525752,50530032,TYMP,-0.401659,-0.665902,-0.591141,0.136790,-0.339752,-3.473521,...,-3.473521,-0.957286,0.838976,-0.042517,0.043358,0.504016,-0.794035,-0.142019,0.655034,0.343038
8509,chr22,50568861,50578465,CPT1B,-0.107367,0.785424,1.182466,-2.177783,0.429245,-2.177783,...,-2.177783,0.120724,0.193813,0.111563,-0.207714,0.588637,0.476276,-2.177783,-2.177783,0.386776
8510,chr22,50578959,50601455,CHKB,0.827567,0.399443,0.343856,-0.444998,-0.388123,-4.293165,...,0.864352,-1.021685,-0.304931,-0.960671,-0.871046,0.947805,0.016035,0.898447,0.728951,0.244254
8511,chr22,50622754,50628173,ARSA,0.036472,-0.469039,-0.415253,-0.003008,-0.121544,-3.581796,...,3.770405,0.071411,0.398842,-0.911205,0.515833,-0.377804,0.097574,-0.496412,-0.550704,0.287421


#### Read genotype_df and vatiant_df

In [8]:
genotype_df = pd.read_parquet('/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/Genetics/WGS_HP_genotype.parquet')
variant_df = pd.read_parquet('/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/Genetics/WGS_HP_variant_df.parquet')
variant_df['chrom'] = 'chr' + variant_df['chrom'].astype(str)

#### Run tensorqtl

In [9]:
#设置bed文件所在的文件夹路径
bed_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/scRNA-seq/06.gene_expression_bed/'
output_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL/'

#获取所有以.bed结尾的文件
bed_files = [f for f in os.listdir(bed_dir) if f.endswith('.bed')]

In [10]:
len(bed_files)

99

In [11]:
# 遍历文件，处理并创建对应文件夹
for bed_file in bed_files:
    cell_name = os.path.splitext(bed_file)[0]  # 去掉.bed后缀
    folder_path = os.path.join(output_dir, cell_name)
    os.makedirs(folder_path, exist_ok=True)  # 创建文件夹

    #read_expression_file and covariates
    expression_bed = f'/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/scRNA-seq/06.gene_expression_bed/{cell_name}.bed'
    covariates_file = f'/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/scRNA-seq/05.PEER/01.factor/{cell_name}.csv'

    #load phenotypes and covariates
    #align phenotypes and covariates
    phenotype_df, phenotype_pos_df = tensorqtl.read_phenotype_bed(expression_bed)
    covariates_df = pd.read_csv(covariates_file, sep=',', index_col=0)
    phenotype_df = phenotype_df[covariates_df.index]

    #decited the number of peer factor
    sample_size = covariates_df.shape[0]
    if sample_size <= 150:
        PF = list('PF' + str(i) for i in range(1, 16))
    elif sample_size <= 250:
        PF = list('PF' + str(i) for i in range(1, 31))
    elif sample_size <= 350:
        PF = list('PF' + str(i) for i in range(1, 46))
    elif sample_size > 350:
        PF = list('PF' + str(i) for i in range(1, 61))
        info = list(['Gender', 'Age', 'Batch', 'Pregnancy', 'PC1', 'PC2'])
        cova_need = info + PF
        covariates_df = covariates_df.loc[:,cova_need]

    #cis-QTL mapping: permutations (对每个基因，寻找在附近（cis-window，一般是±1Mb）的SNP是否与其表达量显著相关。)
    cis_df = cis.map_cis(genotype_df, variant_df, phenotype_df, phenotype_pos_df, covariates_df, 
                         maf_threshold=0.1, seed=8888)

    #后续统一再在py里面算（环境问题）
    #tensorqtl.calculate_qvalues(cis_df)

    #保存
    cis_df.to_csv(f'{folder_path}/all_lead_perm.csv')

    #cis-QTL mapping: summary statistics for all variant-phenotype pairs（所有变异-表型对的统计汇总）
    cis.map_nominal(genotype_df, variant_df, phenotype_df, phenotype_pos_df, 'all', covariates_df, maf_threshold=0.1, 
                    output_dir=f'{folder_path}')

cis-QTL mapping: empirical p-values for phenotypes
  * 815 samples
  * 13189 phenotypes
  * 66 covariates
  * 6990104 variants
  * applying in-sample 0.1 MAF filter
  * cis-window: ±1,000,000
  * using seed 8888
  * checking phenotypes: 13189/13189
    ** dropping 1340 phenotypes without variants in cis-window
  * computing permutations
    processing phenotype 11849/11849
  Time elapsed: 36.91 min
done.
cis-QTL mapping: nominal associations for all variant-phenotype pairs
  * 815 samples
  * 13189 phenotypes
  * 66 covariates
  * 6990104 variants
  * applying in-sample 0.1 MAF filter
  * cis-window: ±1,000,000
  * checking phenotypes: 13189/13189
    ** dropping 1340 phenotypes without variants in cis-window
  * Computing associations
    Mapping chromosome chr1
    processing phenotype 600/11849    time elapsed: 0.17 min
    * writing output
    Mapping chromosome chr2
    processing phenotype 1513/11849    time elapsed: 0.45 min
    * writing output
    Mapping chromosome chr3
    p